# LangChain e LangGraph

O LangChain dá uma interface só para modelo, mensagens e ferramentas, qualquer que seja o provedor por trás. O LangGraph descreve um fluxo como um grafo de estado: nós que são funções sobre um estado compartilhado e arestas que decidem qual nó vem depois.

A primeira parte do notebook percorre as peças do LangChain. A segunda monta grafos, do mais simples até um que escolhe o caminho de acordo com a entrada.

In [ ]:
# No Google Colab, descomente e rode uma vez.
# !pip install -q langchain langchain-openai langgraph

from typing import Annotated, Literal

from IPython.display import Image
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain.tools import tool
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

## LangChain

### Chamando o modelo

A string escolhe o provedor e o modelo. Cada provedor tem um pacote de integração próprio, e todos devolvem um objeto com a mesma interface, de modo que trocar de provedor não muda o resto do código.

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)

# As linhas abaixo trocam o provedor sem mudar o resto do notebook.
# model = init_chat_model("groq:openai/gpt-oss-120b", temperature=0.0)
# model = init_chat_model("ollama:qwen2.5:1.5b", temperature=0.0)

response = model.invoke("Quanto é 2 + 2?")
response

O `invoke` devolve uma `AIMessage` em vez de texto. O texto está em `content`, e o consumo da chamada em `usage_metadata`.

In [ ]:
print(response.content)
print(response.usage_metadata)

O mesmo objeto aceita três chamadas: `invoke` devolve a resposta inteira, `stream` devolve pedaço por pedaço e `batch` resolve uma lista de entradas de uma vez. Essa interface comum é o que o LangChain chama de runnable, e todas as peças que aparecem a seguir a têm.

In [ ]:
for chunk in model.stream("Conte de um a cinco, com os números por extenso."):
    print(chunk.content, end="|")

In [ ]:
questions = ["Qual é a capital da França?", "Qual é a capital do Japão?"]

for answer in model.batch(questions):
    print(answer.content)

### Mensagens

O dicionário `{"role": ..., "content": ...}` continua aceito, e o LangChain acrescenta uma classe por papel. A conversa é uma lista dessas mensagens, e a resposta do modelo entra na lista como o próprio objeto devolvido.

In [ ]:
messages = [
    SystemMessage("Responda em uma frase."),
    HumanMessage("O que é um agente?"),
]

response = model.invoke(messages)
messages.append(response)
messages.append(HumanMessage("E o que ele não é?"))

print(model.invoke(messages).content)

### Templates e composição

Montar o prompt com f-strings funciona até ele ter histórico e várias variáveis. O `ChatPromptTemplate` faz essa montagem sobre uma lista de mensagens, com as variáveis entre chaves, e o `MessagesPlaceholder` reserva o lugar de uma lista inteira de mensagens, que é onde entra o histórico.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente da área de {area}. Responda em uma frase."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

filled = prompt.invoke({
    "area": "astronomia",
    "history": [HumanMessage("Oi"), AIMessage("Olá, como posso ajudar?")],
    "question": "Por que a Lua tem fases?",
})
filled.messages

O operador `|` liga um runnable ao seguinte, passando a saída de um como entrada do próximo, e o resultado é outro runnable. No fim da corrente entra um parser, que lê a `AIMessage` e devolve outro tipo: o `StrOutputParser` devolve o texto, o `JsonOutputParser` devolve um dicionário e o `CommaSeparatedListOutputParser` devolve uma lista.

In [ ]:
chain = prompt | model | StrOutputParser()

print(chain.invoke({"area": "astronomia", "history": [], "question": "Por que a Lua tem fases?"}))

### Saída estruturada

O `with_structured_output` recebe um esquema Pydantic e devolve um runnable que responde com o objeto já validado, em vez de texto para interpretar depois.

In [ ]:
class Event(BaseModel):
    name: str
    place: str
    city: str
    date: str = Field(description="Data no formato AAAA-MM-DD")


structured_model = model.with_structured_output(Event)
structured_model.invoke("A palestra sobre visão computacional acontece no auditório do IMD, em Natal, no dia 15 de outubro de 2026.")

### Ferramentas

O decorador `tool` lê a assinatura e a docstring para montar o esquema que descreve a ferramenta ao modelo. Ele devolve um objeto em vez da função, e a execução passa por `invoke`.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Avalia uma expressão aritmética, como 12 * (3 + 4)."""
    return str(eval(expression, {"__builtins__": {}}, {}))


print(calculate.name)
print(calculate.description)
print(calculate.args)
print(calculate.invoke({"expression": "4871 * 3926"}))

O `bind_tools` liga as ferramentas ao modelo. A chamada volta em `tool_calls`, já estruturada e com um `id`, sem nenhum texto para interpretar.

In [ ]:
model_with_tools = model.bind_tools([calculate])

question = "Quanto é 4871 vezes 3926?"
response = model_with_tools.invoke(question)
response.tool_calls

Passar a chamada inteira para `invoke` executa a ferramenta e devolve uma `ToolMessage` com o `tool_call_id` preenchido. O ciclo fecha em quatro mensagens: pergunta, chamada, observação e resposta.

In [ ]:
observation = calculate.invoke(response.tool_calls[0])
print(observation)

print(model_with_tools.invoke([HumanMessage(question), response, observation]).content)

### Exercício 1

Escreva uma ferramenta com `@tool` e ligue-a ao modelo com `bind_tools`. Monte uma corrente de três peças com `|`: um `ChatPromptTemplate` com uma variável para o pedido, o modelo com a ferramenta e um `StrOutputParser`. Rode com um pedido que exija a ferramenta. O que o parser devolveu, e onde ficou a chamada da ferramenta?

In [ ]:
# Seu código aqui

## LangGraph

Um grafo tem um estado compartilhado, nós que são funções sobre esse estado e arestas que dizem qual nó vem depois. Descrever um fluxo assim separa o que cada passo faz da regra que decide a ordem dos passos, e deixa a topologia visível no código.

### Estado, nós e arestas

O estado é um `TypedDict`. Cada nó recebe o estado inteiro e devolve um dicionário só com as chaves que quer atualizar. O grafo abaixo não usa modelo nenhum, para que o mecanismo apareça sozinho.

In [ ]:
class State(TypedDict):
    text: str
    words: int


def clean(state: State) -> dict:
    """Remove espaços extras do texto."""
    return {"text": " ".join(state["text"].split())}


def count(state: State) -> dict:
    """Conta as palavras do texto."""
    return {"words": len(state["text"].split())}

O grafo é montado com o estado, recebe os nós pelo nome e as arestas entre eles. `START` e `END` são os dois nós que todo grafo já tem. Compilar transforma a descrição em um runnable.

In [ ]:
builder = StateGraph(State)
builder.add_node("clean", clean)
builder.add_node("count", count)
builder.add_edge(START, "clean")
builder.add_edge("clean", "count")
builder.add_edge("count", END)

graph = builder.compile()
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
graph.invoke({"text": "  inteligência   artificial  é   legal "})

A entrada é um estado parcial e a saída é o estado final. O `clean` substituiu `text`, e o `count` leu o texto já limpo e escreveu `words`. Cada nó só mexe no que devolve.

### Um nó com modelo

Um nó que chama o modelo é uma função como as outras: recebe o estado, faz o trabalho e devolve as chaves que mudou. O estado abaixo carrega a conversa em uma lista comum, e o nó devolve a resposta do modelo.

In [ ]:
class PlainState(TypedDict):
    messages: list


def respond(state: dict) -> dict:
    """Chama o modelo com a conversa inteira e devolve a resposta."""
    return {"messages": [model.invoke(state["messages"])]}

In [ ]:
plain_builder = StateGraph(PlainState)
plain_builder.add_node("respond", respond)
plain_builder.add_edge(START, "respond")
plain_builder.add_edge("respond", END)
plain_chat = plain_builder.compile()

replaced = plain_chat.invoke({"messages": [HumanMessage("O que é um agente? Responda em uma frase.")]})

print(len(replaced["messages"]))
for message in replaced["messages"]:
    print(message.type, "|", message.content)

Sobrou uma mensagem só, e é a resposta. O nó devolveu `{"messages": [resposta]}`, e por padrão a atualização de um nó substitui o valor da chave, então a pergunta foi embora junto. Um laço que precisa do histórico não sobrevive a isso.

### Exercício 2

Monte um grafo de dois nós em sequência: o primeiro resume um parágrafo em uma frase com o modelo, o segundo grava no estado quantos caracteres o resumo economizou. Use um parágrafo qualquer como entrada. Quantos por cento do texto original o resumo ocupou?

In [ ]:
# Seu código aqui

### Reducer

Um reducer é a regra que junta o que o nó devolveu ao que já estava no estado, declarada na anotação da chave. Ele é uma função que recebe o valor atual e a atualização e devolve o novo valor.

In [ ]:
current = [HumanMessage("Oi")]
update = [AIMessage("Olá, como posso ajudar?")]

add_messages(current, update)

O `add_messages` é uma função comum: recebe a lista que está no estado e a lista que o nó devolveu, e entrega as duas concatenadas. Ele também atribui um `id` a cada mensagem, e uma atualização que traga um `id` já existente substitui aquela mensagem em vez de acrescentar outra.

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


chat_builder = StateGraph(ChatState)
chat_builder.add_node("respond", respond)
chat_builder.add_edge(START, "respond")
chat_builder.add_edge("respond", END)
chat_graph = chat_builder.compile()

appended = chat_graph.invoke({"messages": [HumanMessage("O que é um agente? Responda em uma frase.")]})

print(len(appended["messages"]))
for message in appended["messages"]:
    print(message.type, "|", message.content[:60])

O nó é o mesmo e o grafo tem a mesma forma; mudou só a anotação da chave `messages`. A conversa termina com as duas mensagens, e é isso que permite um laço acrescentar mensagens a cada volta sem apagar o que veio antes.

### Arestas condicionais

Uma aresta condicional é uma função que lê o estado e devolve o nome do próximo nó. O grafo abaixo classifica o pedido e manda cada tipo para um nó diferente.

In [ ]:
class Route(BaseModel):
    kind: Literal["conta", "conversa"]
    expression: str = Field(default="", description="A expressão aritmética, se o pedido for uma conta")


class RouterState(TypedDict):
    question: str
    kind: str
    expression: str
    answer: str


def classify(state: RouterState) -> dict:
    """Decide se o pedido é uma conta ou uma conversa."""
    route = model.with_structured_output(Route).invoke(state["question"])
    return {"kind": route.kind, "expression": route.expression}


In [ ]:
def compute(state: RouterState) -> dict:
    """Resolve a conta com a ferramenta."""
    return {"answer": calculate.invoke({"expression": state["expression"]})}


def chat(state: RouterState) -> dict:
    """Responde com o modelo."""
    return {"answer": model.invoke(state["question"]).content}


def route(state: RouterState) -> Literal["compute", "chat"]:
    """Escolhe o próximo nó pelo tipo do pedido."""
    return "compute" if state["kind"] == "conta" else "chat"


In [ ]:
builder = StateGraph(RouterState)
builder.add_node("classify", classify)
builder.add_node("compute", compute)
builder.add_node("chat", chat)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, ["compute", "chat"])
builder.add_edge("compute", END)
builder.add_edge("chat", END)

router = builder.compile()
Image(router.get_graph().draw_mermaid_png())

In [ ]:
for question in ["Quanto é 4871 vezes 3926?", "Quem escreveu Dom Casmurro?"]:
    result = router.invoke({"question": question})
    print(result["kind"], ":", result["answer"])

As linhas tracejadas do desenho são as arestas condicionais. A lista passada em `add_conditional_edges` diz quais destinos a função pode devolver, e serve para o desenho e para a validação do grafo.

### Exercício 3

Escreva um grafo que classifique cada e-mail em `reclamacao` ou `orcamento` e mande cada classe para um nó diferente: um resume o problema relatado, o outro extrai os itens e as quantidades pedidas. Os dois terminam no mesmo nó `draft`, que redige a resposta a partir do que o ramo anterior deixou no estado. Qual nó cada um dos dois e-mails acionou?

In [ ]:
EMAILS = [
    "Comprei a cafeteira no dia 3 e ela parou de esquentar na primeira semana. Quero uma solução.",
    "Bom dia. Preciso de 12 cadeiras e 3 mesas para montar um escritório. Fecham orçamento?",
]

# Seu código aqui